In [1]:
import sys
sys.path.append('/home/galadriel/dr_benchmark/dev')
from kg_processing import load_knowledge_graph, save_knowledge_graph
import my_knowledge_graph
import torch

/home/galadriel/dr_benchmark/dev/my_data_redundancy.py:22: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
config = {'common': {'input_pkl': '/home/galadriel/dr_benchmark/DL1experiment/withoutDL1/kg_processing/kg.pkl',
                     'out': '/home/galadriel/dr_benchmark/DL1experiment/withoutDL1/kg_processing_indication/'}}

In [3]:
kg_train, kg_val, kg_test = load_knowledge_graph(config)

2025-04-09 09:41:45,876 - root - INFO - Will not run the preparation step. Using KG stored in: /home/galadriel/dr_benchmark/DL1experiment/withoutDL1/kg_processing/kg.pkl
/home/galadriel/anaconda3/envs/benchmark/lib/python3.9/site-packages/torch/storage.py:414: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where 

In [4]:
def reassign_non_target_to_train(kg_train, kg_val, kg_test, target_rel_name):
    target_rel_id = kg_train.rel2ix[target_rel_name]  # même dico pour tous les splits

    def filter_target(kg, target_rel_id):
        is_target = kg.relations == target_rel_id
        return is_target, ~is_target

    # Filtrage des triplets target vs non-target dans val et test
    val_target_mask, val_non_target_mask = filter_target(kg_val, target_rel_id)
    test_target_mask, test_non_target_mask = filter_target(kg_test, target_rel_id)

    # Construire le nouveau train
    new_heads = torch.cat([
        kg_train.head_idx,
        kg_val.head_idx[val_non_target_mask],
        kg_test.head_idx[test_non_target_mask]
    ])
    new_tails = torch.cat([
        kg_train.tail_idx,
        kg_val.tail_idx[val_non_target_mask],
        kg_test.tail_idx[test_non_target_mask]
    ])
    new_rels = torch.cat([
        kg_train.relations,
        kg_val.relations[val_non_target_mask],
        kg_test.relations[test_non_target_mask]
    ])

    new_train = my_knowledge_graph.KnowledgeGraph(
        kg={'heads': new_heads, 'tails': new_tails, 'relations': new_rels},
        ent2ix=kg_train.ent2ix,
        rel2ix=kg_train.rel2ix,
        dict_of_heads=kg_train.dict_of_heads,
        dict_of_tails=kg_train.dict_of_tails,
        dict_of_rels=kg_train.dict_of_rels
    )

    new_val = my_knowledge_graph.KnowledgeGraph(
        kg={
            'heads': kg_val.head_idx[val_target_mask],
            'tails': kg_val.tail_idx[val_target_mask],
            'relations': kg_val.relations[val_target_mask]
        },
        ent2ix=kg_train.ent2ix,
        rel2ix=kg_train.rel2ix,
        dict_of_heads=kg_train.dict_of_heads,
        dict_of_tails=kg_train.dict_of_tails,
        dict_of_rels=kg_train.dict_of_rels
    )

    new_test = my_knowledge_graph.KnowledgeGraph(
        kg={
            'heads': kg_test.head_idx[test_target_mask],
            'tails': kg_test.tail_idx[test_target_mask],
            'relations': kg_test.relations[test_target_mask]
        },
        ent2ix=kg_train.ent2ix,
        rel2ix=kg_train.rel2ix,
        dict_of_heads=kg_train.dict_of_heads,
        dict_of_tails=kg_train.dict_of_tails,
        dict_of_rels=kg_train.dict_of_rels
    )

    return new_train, new_val, new_test


In [5]:
new_train, new_val, new_test = reassign_non_target_to_train(kg_train, kg_val, kg_test, 'indication')  

In [6]:
assert kg_train.dict_of_heads == new_train.dict_of_heads == new_val.dict_of_heads == new_test.dict_of_heads == kg_val.dict_of_heads == kg_test.dict_of_heads

In [7]:
save_knowledge_graph(config, new_train, new_val, new_test)

2025-04-08 11:10:49,425 - root - INFO - Saving results to /home/galadriel/dr_benchmark/DL1experiment/withoutDL1/kg_processing_indication/kg.pkl...
